<a href="https://colab.research.google.com/github/Gayathri-rfr/LangchainLearner/blob/main/ResearchAssist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ddgs langchain-community langchain-huggingface


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
!pip install langchain-openai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 15.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19


In [3]:
!pip install -q openai

In [5]:
from langchain_openai import ChatOpenAI,OpenAI
from google.colab import userdata
import os
from openai import OpenAI
#os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
token = userdata.get('GITHUB_TOKEN')
os.environ["OPENAI_API_KEY"] = token

'''llm = ChatOpenAI(
   model="gpt-4o-mini",
    temperature =0,
    )'''
'''
client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=token,
)
response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "New developements in AI for drug discovery",
        }
    ],
    model="gpt-4o", # You can also try "meta-llama-3-70b-instruct"
    temperature=1,
    max_tokens=4096,
    top_p=1
)
print(response.choices[0].message.content)'''
'''
llm = ChatOpenAI(
   model="gpt-4o-mini",
   api_key=token,
    temperature =0,
   base_url="https://models.inference.ai.azure.com"
    )
response = llm.invoke("Explain quantum entanglement in one sentence.")
print(response)'''

content='Quantum entanglement is a phenomenon where two or more particles become interconnected in such a way that the state of one particle instantaneously influences the state of the other, regardless of the distance separating them.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 16, 'total_tokens': 57, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DQD0SYGgaoDTH6rqZJbxNyYmAfuK5', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d4e85-f672-7050-b430-dbcb64a94b94-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 16, 'output_tokens': 41, 'total_tokens': 57, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'ou

In [8]:
#rint(response.content)
!pip install duckduckgo-search

In [20]:
import os
from re import search
from typing import Annotated
from typing_extensions import TypedDict
from duckduckgo_search import DDGS
from langgraph.graph import add_messages, START, StateGraph, END
from google.colab import userdata
from langchain_huggingface import HuggingFaceEndpoint
from langchain_openai import ChatOpenAI

class ResearchAssist(TypedDict):
  messages : Annotated[list, add_messages]
  topic : str
  report_format : str
  search_queries : list[str]
  raw_results : list[str]
  processed_results : list[str]
  report : str
  score : int
  attempt : int


#token = userdata.get('HF_TOKEN')

llm = ChatOpenAI(
   model="gpt-4o-mini",
   api_key=token,
    temperature =0,
   base_url="https://models.inference.ai.azure.com"
    )

def plan_node(ResearchAssist):
  query = ResearchAssist['topic']
  prompt = f''' convert the following user input topic into an effective web search query.
  keep it consise and specific.

  User input topic = {query}
  '''

  response = llm.invoke(prompt)

  return {
      "search_queries" : [response.content.strip()]
  }

def searchweb_node(ResearchAssist):
  query = ResearchAssist['search_queries']
  result_list = []
  try :
    with DDGS() as ddgs:
      # The query is a list, so we need to access the string element directly
      results = list(ddgs.text(query[0],max_results = 3))
      for r in results:
        formatted = f"Title: {r.get('title', '')}\nSnippet: {r.get('body', '')}"
        result_list.append(formatted)
  except Exception as e:
    result_list.append(f"Web search error : {str(e)}")

  return {
      "raw_results" : result_list
       }

def filter_result_node(ResearchAssist):
  query= ResearchAssist['topic']
  results = ResearchAssist['raw_results']

  prompt = f''' Filter the raw results to meaningful and high quality information
                Topic = {ResearchAssist['topic']}
                Results = {chr(10).join(ResearchAssist['raw_results'])}'''

  response = llm.invoke(prompt)
  filtered =[r.strip() for r in response.content.split('\n') if r.strip()]
  return { "processed_results" : filtered}


builder = StateGraph(ResearchAssist)
builder.add_node("plan",plan_node)
builder.add_node("search",searchweb_node)
builder.add_node("filter",filter_result_node)

builder.set_entry_point("plan")
builder.add_edge("plan","search")
builder.add_edge("search","filter")
builder.add_edge("filter",END)
agent = builder.compile()

#  return builder.build()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [22]:
initial_state = {
    "topic": "latest advancements in AI for drug discovery",
    "messages": [],
    "report_format": "summary", # You can specify other formats like 'detailed', 'bullet_points', etc.
    "search_queries": [],
    "raw_results": [],
    "processed_results": [],
    "report": "",
    "score": 0,
    "attempt": 0
}

# Invoke the agent with the initial state
result = agent.invoke(initial_state)

# You can then inspect the result
print(result['processed_results'])

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac

['Here are some meaningful and high-quality insights on the latest advancements in AI for drug discovery:', '1. **Machine Learning Models**: Recent advancements in machine learning algorithms, particularly deep learning, have significantly improved the ability to predict molecular interactions and drug efficacy. Techniques such as convolutional neural networks (CNNs) and recurrent neural networks (RNNs) are being utilized to analyze complex biological data.', '2. **Generative Models**: Generative models, including Generative Adversarial Networks (GANs) and Variational Autoencoders (VAEs), are being employed to design novel drug candidates. These models can generate new molecular structures with desired properties, accelerating the lead optimization process.', '3. **Natural Language Processing (NLP)**: NLP techniques are being used to mine vast amounts of scientific literature and clinical trial data. This helps researchers identify potential drug targets and understand the mechanisms o

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [27]:
print(result['processed_results'])

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


['Here are some meaningful and high-quality insights on the latest advancements in AI for drug discovery:', '1. **Machine Learning Models**: Recent advancements in machine learning algorithms, particularly deep learning, have significantly improved the ability to predict molecular interactions and drug efficacy. Techniques such as convolutional neural networks (CNNs) and recurrent neural networks (RNNs) are being utilized to analyze complex biological data.', '2. **Generative Models**: Generative models, including Generative Adversarial Networks (GANs) and Variational Autoencoders (VAEs), are being employed to design novel drug candidates. These models can generate new molecular structures with desired properties, accelerating the lead optimization process.', '3. **Natural Language Processing (NLP)**: NLP techniques are being used to mine vast amounts of scientific literature and clinical trial data. This helps researchers identify potential drug targets and understand the mechanisms o

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [46]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
print("Deprecation warnings from jupyter_client suppressed.")

Deprecation warnings from jupyter_client suppressed.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [48]:
import gradio as gr

def display_results(new_topic):
    # Update the topic in the initial state
    initial_state["topic"] = new_topic

    # Invoke the agent with the updated initial state
    result = agent.invoke(initial_state)

    if 'processed_results' in result:
        output = "\n".join(result['processed_results'])
    else:
        output = "No processed results found for this topic."
    return output

# Create a Gradio Interface
interface = gr.Interface(
    fn=display_results,
    inputs=gr.Textbox(lines=1, label="Enter Research Topic", value="", interactive=True),
    outputs=gr.Textbox(lines=20, label="Search Results.."),
    title="Your Research Assistant",
    description="Enter a topic to generate a research report using the LangGraph agent."
)

# Launch the Gradio interface
interface.launch(inline=True, share=True)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


* Running on public URL: https://678449c4be9c5de072.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
